In [291]:
import pyreadstat as rs
import os
import DE_Utilities as DE

In [292]:
data = rs.read_xport("SMQ_L.xpt")

In [293]:
# DE.xpt_to_csv("datasets" , "datasets_csv")

In [294]:
# DE.join_datasets("datasets_csv" , output_dir = "silver" , first_file="ACQ_L.csv" , join_on='SEQN')

In [295]:
import pandas as pd 

In [296]:
df = pd.read_csv("datasets_csv/HIQ_L.csv")

In [297]:
df_2 = pd.read_csv("datasets_csv/HSQ_L.csv")

In [298]:
df.columns

Index(['Unnamed: 0', 'SEQN', 'HIQ011', 'HIQ032A', 'HIQ032B', 'HIQ032C',
       'HIQ032D', 'HIQ032E', 'HIQ032F', 'HIQ032H', 'HIQ032I', 'HIQ210'],
      dtype='str')

In [299]:
df.merge(df_2 , on = "SEQN" , how = "outer")

,Unnamed: 0_x,SEQN,HIQ011,HIQ032A,HIQ032B,HIQ032C,HIQ032D,HIQ032E,HIQ032F,HIQ032H,HIQ032I,HIQ210,Unnamed: 0_y,HSQ590
0,0,130378.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0.0,NaN
1,1,130379.0,1.0,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,2.0,1.0,1.0
2,2,130380.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,8.0,NaN,1.0,2.0,1.0
3,3,130381.0,1.0,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,2.0,NaN,NaN
4,4,130382.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11928,11928,142306.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11929,11929,142307.0,1.0,NaN,2.0,NaN,4.0,NaN,NaN,NaN,NaN,2.0,6611.0,1.0
11930,11930,142308.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,6612.0,2.0
11931,11931,142309.0,1.0,1.0,NaN,NaN,NaN,NaN,6.0,NaN,NaN,2.0,6613.0,1.0


In [300]:
df = pd.read_csv("silver/final_data.csv")

In [301]:
df = df.select_dtypes(exclude="str")

In [302]:
m , n = df.shape

In [303]:
m

11933

In [304]:
fil = (df > 0).astype("int").sum(axis = 0) > 0.80 * m

In [305]:
cols = df.loc[: , fil].columns

In [306]:
required_cols = ['BPQ020' , 'RIAGENDR' , "RIDAGEYR"  , "BMXBMI", "PAD790Q" , "DIQ010" , "DR1TSODI" ] 
required_cols_mapping = ["had_hypertension" , "gender" , "age_at_years"   , "Body_Mass_Index"   , "Frequency_of_moderate_LTPA" , "had_diabetes"  , "Sodium_(mg)_perday"]

In [307]:
df_f_v_1 = df.loc[: , required_cols]

In [308]:
df_f_v_1.columns = required_cols_mapping

In [309]:
df_f_v_2 = df_f_v_1[~df_f_v_1.had_hypertension.isna()]

In [310]:
df_f_v_3 = df_f_v_2

In [311]:
BMI_mean = df_f_v_3.Body_Mass_Index.mean()

In [312]:
df_f_v_4 = df_f_v_3.fillna(BMI_mean)

In [313]:
df_f_v_5 = df_f_v_4[(df_f_v_4.had_hypertension == 1.0) | (df_f_v_4.had_hypertension == 2.0)]


In [314]:
df_f_v_5.had_hypertension = df_f_v_5.had_hypertension.apply(lambda row : 1 if row == 1.0 else 0)
df_f_v_5.gender = df_f_v_5.gender.apply(lambda row : 1 if row == 1.0 else 0)

In [315]:
df_f_v_6 = df_f_v_5.loc[(df_f_v_5.had_diabetes == 1.0)|(df_f_v_5.had_diabetes == 2.0)]

In [316]:
df_f_v_6.had_diabetes = df_f_v_6.had_diabetes.apply(lambda row : 1 if row == 1.0 else 0)

In [324]:
df_f_v_final = df_f_v_6.rename(columns = {"had_hypertension" : "target"})

In [ ]:
def transform(df):
    required_cols = ['BPQ020' , 'RIAGENDR' , "RIDAGEYR"  , "BMXBMI", "PAD790Q" , "DIQ010" , "DR1TSODI" ] 
    required_cols_mapping = ["had_hypertension" , "gender" , "age_at_years"   , "Body_Mass_Index"   , "Frequency_of_moderate_LTPA" , "had_diabetes"  , "Sodium_(mg)_perday"]
    df_f_v_1 = df.loc[: , required_cols]
    df_f_v_1.columns = required_cols_mapping
    df_f_v_2 = df_f_v_1[~df_f_v_1.had_hypertension.isna()]
    df_f_v_3 = df_f_v_2
    BMI_mean = df_f_v_3.Body_Mass_Index.mean()
    df_f_v_4 = df_f_v_3.fillna(BMI_mean)
    df_f_v_5 = df_f_v_4[(df_f_v_4.had_hypertension == 1.0) | (df_f_v_4.had_hypertension == 2.0)]
    df_f_v_5.had_hypertension = df_f_v_5.had_hypertension.apply(lambda row : 1 if row == 1.0 else 0)
    df_f_v_5.gender = df_f_v_5.gender.apply(lambda row : 1 if row == 1.0 else 0)
    df_f_v_6 = df_f_v_5.loc[(df_f_v_5.had_diabetes == 1.0)|(df_f_v_5.had_diabetes == 2.0)]
    df_f_v_6.had_diabetes = df_f_v_6.had_diabetes.apply(lambda row : 1 if row == 1.0 else 0)
    df_f_v_final = df_f_v_6.rename(columns = {"had_hypertension" : "target"})
    X = df_f_v_final.iloc[: , 1:]
    Y = df_f_v_final.iloc[: , 0]

    return df_f_v_final , X , Y
    


In [ ]:
def load(destinations , data):
    
    if len(destinations) != len(data):
        logger.error("The length of destinations is not equal to length of data")
        return None
    logger.info(f"The data is being loaded to destinations")
    try:
        i = 0 
        while i < len(destinations):
            df = data[i]
            dest = destinations[i]
            df.to_csv(dest)
            i +=1
        logger.info(f"The data has been loaded to destinations successfully")
    except Exception as e:
        logger.error(f"The data hasn't been loaded successfully , Error : {e}")
    return None

    